[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhmk/hocmaytrongtaichinh/blob/main/Module_3/L3.ipynb)

## HỌC MÁY TRONG TÀI CHÍNH


MODULE 3 | BÀI 3


---

# **CÂY QUYẾT ĐỊNH**

|  |  |
|:---|:---|
|**Thời gian đọc** |  90 phút |
|**Kiến thức nền** | Hồi quy logistic  |
|**Từ khóa** | Tạp chất Gini, Nút, Nút đồng nhất, Boosting, Đường cong ROC |


---

*Ta đã xem một trong những công cụ phân loại, đó là phân loại Naïve Bayes. Phương pháp này xây dựng trên khung thống kê Bayes và dùng phân phối tiên nghiệm và phân phối hợp lý. Các phân phối này dựa trên dữ liệu ta huấn luyện mô hình, và ta đưa ra giả định ngây thơ về sự độc lập giữa các biến dự báo. Thường thì các bài toán thực tế không như vậy. Bây giờ ta xem một phương pháp thay thế không đưa ra giả định đó và được dùng rộng rãi trong thực tế.*

## **1. Cây phân loại**

Để hiểu các mô hình dựa trên cây, ta cần hiểu các khối cơ bản tạo nên chúng, là cây quyết định. Các thành phần của cây quyết định được thể hiện ở hình 1. Đó là các nút có tên riêng dựa trên vị trí của chúng trong lưu đồ. Theo dòng chảy từ trên xuống dưới, đầu tiên ta có nút gốc, rồi các nút trong, và cuối cùng là các nút lá. Các nhánh chia sang hướng trái/phải gọi là **nhánh** (branch).

**Hình 1:** Các thành phần cơ bản và cấu trúc của một cây quyết định.

![](https://raw.githubusercontent.com/thanhmk/hocmaytrongtaichinh/main/Module_3/M3L3_h1.png)

Nguồn: python-course.edu, Decision trees in python, tháng 6/2022. https://python-course.eu/machine-learning/decision-trees-in-python.php

Để hình dung cách áp dụng cây quyết định, ta xét một ví dụ về rủi ro vỡ nợ tín dụng đối với khoản vay ngân hàng, chỉ dựa trên ba biến giải thích. Các nhóm của biến mục tiêu là 0: không vỡ nợ, 1: vỡ nợ. Hãy nhớ rằng ví dụ này nhằm truyền đạt ý tưởng của cây quyết định, vì thực tế ngành ngân hàng dùng nhiều biến giải thích/biến dự báo hơn nhiều.

Biến dự báo thứ nhất trong ba biến là một biến nhị phân đánh dấu người nộp đơn có hơn hai năm lịch sử tín dụng trong cơ quan tín dụng hay không. Điều này có giá trị với ngân hàng vì cho phép ngân hàng đánh giá khách hàng hoạt động tài chính ra sao. Biến dự báo thứ hai cũng là biến nhị phân, bằng 1 nếu khách hàng có hơn 1 lần chậm thanh toán ở các khoản vay khác trong 2 năm qua và 0 nếu không. Biến dự báo thứ ba là thu nhập hằng tháng khả dụng của khách hàng. Giả sử đơn vị tiền tệ là rand Nam Phi. Hình này xác định các quy tắc dẫn ta đến các nhóm mục tiêu cụ thể. Chẳng hạn, hình 2 cho thấy những người nộp đơn có hơn hai năm lịch sử tín dụng có xác suất vỡ nợ (PD) là 9%, khi nhìn vào đường đi dẫn đến nút lá dưới cùng bên trái.

**Hình 2:** Mô hình cây quyết định áp dụng cho dữ liệu hồ sơ tín dụng với 3 biến dự báo.

![](https://raw.githubusercontent.com/thanhmk/hocmaytrongtaichinh/main/Module_3/M3L3_h2.png)

Nhóm dân số này chiếm 60% tổng thể. Đi theo đường sang bên phải, ta xét những người có tối đa hai năm lịch sử tín dụng. Nhóm này có PD là 45%. Tách tiếp nhóm này thành những người có hơn 1 lần chậm trả trong lịch sử tín dụng hai năm và có thu nhập dưới 20.000 rand, có 79% khả năng vỡ nợ. Nhóm này chiếm 20% tổng thể và có khả năng vỡ nợ cao. Điều quan trọng là phải đánh giá tính logic và tính hợp lý của các quy tắc, vì thường nhà phân tích sẽ cần giải thích điều này cho các bên liên quan. Trong ví dụ này, dữ liệu xác nhận nghi ngờ của ta rằng những người có thu nhập thấp, ít lịch sử tín dụng và có lịch sử chậm trả hơn một lần có khả năng vỡ nợ.

Giờ ta đã hiểu cách dùng cây quyết định, ta xem cách cây được huấn luyện trên dữ liệu. Để làm vậy, trước hết ta xem xét một số thước đo thường dùng để xác định cách chia ở các nhánh.

## **2. Tạp chất Gini**

Cây quyết định chia các mẫu bằng cách chọn lần lượt các biến dự báo theo một thước đo được tính. Thước đo ta xét là **tạp chất Gini** (Gini impurity), đặt theo tên Corrado Gini. Nó được cho bởi

$$G = 1-\sum_{k}(p_k)^2\tag{3.1}$$

trong đó tổng lấy trên mọi nhóm mục tiêu có thể có $k=1,\ldots,c$. Trong ví dụ của ta, điều này thấy ở hình 2 với $k=1,2$. Các số hạng $p_k$ là tỷ lệ của nhóm trong nút cụ thể đó. Chẳng hạn, hình 2 chia biến có hơn hai năm lịch sử tín dụng trước. Đặc trưng này chỉ có hai nhóm và ta có tỷ lệ vỡ nợ của các nhóm ở bảng 1.

**Bảng 1**

| Lịch sử tín dụng hơn 2 năm | % vỡ nợ |
|:---:|:---:|
| CÓ | 9% |
| KHÔNG | 91% |

Do đó từ (3.1) ta có,

$$
\begin{align*}
G &= 1-[(0.09^2)+(0.91^2)]\\
&= 0.164
\end{align*}
$$

Ý tưởng đằng sau tạp chất Gini là thu được nút càng thuần càng tốt, qua đó giảm thiểu rủi ro phân loại sai. Ta muốn tạp chất Gini càng thấp càng tốt để đạt điều đó. Bạn có thể thấy nút lá ngoài cùng bên trái có tỷ lệ dân số vỡ nợ rất thấp. Tương tự, nút trong của nhóm có lịch sử tín dụng từ 2 năm trở xuống có tạp chất Gini là 0,495. Hai nút lá cuối cùng bên phải có tạp chất Gini như sau:

**Bảng 2**

| Nút lá | Tạp chất Gini |
|:---|:---:|
| ≤ 2 năm lịch sử, hơn 1 lần chậm trả, thu nhập ít nhất 20k rand | 18% |
| ≤ 2 năm lịch sử, hơn 1 lần chậm trả, và thu nhập < 20k rand | 33% |

Biến ở nút gốc được chọn để tạo ra nút thuần nhất, tức là có tạp chất Gini thấp nhất. Do đó cây quyết định phải thử chia trên mọi biến dự báo để xác định điều này.

Một thước đo thường dùng khác để chia cây là entropy. Entropy hoặc tạp chất Gini được dùng để cực đại hóa **lượng thông tin thu được** (information gain). Bạn đọc quan tâm đến phần này có thể xem Hull (133). Ta đã xem cây trong bài toán phân loại, nhưng cần nhớ rằng chúng cũng có thể dùng cho biến mục tiêu liên tục. Đó là các bài toán hồi quy, được trình bày tiếp theo.

## **3. Cây hồi quy**

Đến giờ, ta mới đề cập đến các bài toán phân loại; tuy nhiên trong thực tế ta có thể gặp biến mục tiêu liên tục. Trong trường hợp này, ta sẽ quan tâm đến việc dự báo một lượng hay một giá trị trong một bài toán tài chính. Với loại bài toán này, cây hồi quy là phù hợp. Điểm khác trong biểu đồ so với hình 2 là các nút lá sẽ thể hiện giá trị trung bình của biến mục tiêu. Dữ liệu được chia theo từng biến dự báo ở mọi điểm, và cách chia có biến thiên thấp nhất giữa giá trị dự báo và giá trị mục tiêu thực được chọn làm điểm chia. Thước đo biến thiên là tổng bình phương, vốn quen thuộc từ hồi quy tuyến tính. Với một biến dự báo bất kỳ $j$, ta có (3.2):

$$SSE=\sum_{i\in S_1}(y_i-\hat{y}_1)^2+\sum_{i\in S_2}(y_i-\hat{y}_2)^2\tag{3.2}$$

trong đó $y_i$ là giá trị dự báo, còn $\hat{y}_1$ và $\hat{y}_2$ là giá trị trung bình của phía trái và phía phải của cách chia. Cách chia cho SSE thấp nhất sẽ được chọn. Cây, giống các thuật toán học máy khác, dễ bị quá khớp; tuy nhiên có những cách để giảm thiểu điều này. Một trong số đó gọi là **cắt tỉa** (pruning).

Cắt tỉa nhằm giảm kích thước của cây quyết định, qua đó giảm quá khớp. Việc này được thực hiện bằng cách cắt giảm những phần không quan trọng của cây. Lý tưởng là ta muốn cây khái quát hóa tốt cho các mẫu chưa thấy. Một chiến lược cắt tỉa là tạo ra một cây có các mẫu nhỏ ở nút lá, từ đó giảm độ sâu tối đa của cây và loại bỏ những nút lá bị quá khớp. Đây là ví dụ của cắt tỉa sau (post-pruning). Với cắt tỉa trước (pre-pruning), ta chỉ định các tham số để ngăn các nút lá có quá ít mẫu; ví dụ là độ sâu tối đa hoặc Gini tối thiểu. Việc dừng sớm này có thể hiệu quả vì cần ít tính toán hơn. Hình 3 minh họa cây chưa cắt tỉa và cây đã cắt tỉa.

**Hình 3:** Cây chưa cắt tỉa và cây đã cắt tỉa

![](https://raw.githubusercontent.com/thanhmk/hocmaytrongtaichinh/main/Module_3/M3L3_h3.png)

Một trong những thuật toán phân loại dự báo đơn giản nhất là hồi quy logistic, và cần nhắc đến những so sánh với cây quyết định.

## **4. Hồi quy logistic và cây quyết định**

Cả hai phương pháp đều giải quyết bài toán phân loại và đều dễ diễn giải. Cây quyết định có thể chia dữ liệu một cách phi tuyến; do đó dữ liệu không cần phải phân tách tuyến tính.

**Hình 4:** Dữ liệu không phân tách tuyến tính được

![](https://raw.githubusercontent.com/thanhmk/hocmaytrongtaichinh/main/Module_3/M3L3_h4.png)

Hình 4 minh họa dữ liệu không phân tách tuyến tính được, trong đó cây quyết định dự báo nhóm hiệu quả hơn. Hồi quy logistic xử lý tốt dữ liệu phân tách tuyến tính được. Cây quyết định chia các biến dự báo một cách tuần tự; do đó cũng không cần co giãn thang đo dữ liệu, khác với hồi quy logistic. Hồi quy logistic cũng nhạy với giá trị ngoại lai và độ lệch của dữ liệu, còn cây quyết định thì không. Với ít biến trong cây quyết định, sự tương tác giữa các biến dễ diễn giải và thể hiện. Rất thường khi mô hình cây quyết định được chọn thay vì mô hình hồi quy logistic; tuy nhiên một số tổ chức tài chính bị quy định phải dùng mô hình hồi quy logistic.

Ở phần tiếp theo, ta xem một thuật toán dựa trên cây được cải tiến, tận dụng các cây khác để cải thiện hiệu suất dự báo.

## **5. Boosting**

**Boosting** là phương pháp dựa trên cây, kết hợp sức mạnh của các cây khác, cụ thể là các cây học yếu mà ta gọi là **bộ học yếu** (weak learner). Các bộ học yếu này về cơ bản chỉ có một nút gốc và một lần chia theo biến dự báo đó. Quy trình kết hợp các bộ học yếu này theo cách tuần tự, sao cho mỗi bộ học yếu cải thiện so với bộ học trước, nghĩa là mô hình cuối cùng được cải thiện dần theo thời gian. Chính vì tính tuần tự này mà việc huấn luyện mất nhiều thời gian hơn các phương pháp song song khác. Giả sử ở vòng lặp $i$ ta có một tập dự báo $P_i(X)$ từ một bộ học yếu dựa trên tập biến dự báo $X$. Các dự báo ở các vòng lặp có thể liên hệ với nhau như sau:

$$P_{i+1}(X)=P_i(X)+T_i(X)$$

trong đó $T_i(X)$ là bộ học dựa trên cây được đưa vào ở vòng lặp $i$ để cải thiện $P_i(X)$. Phần dư được xem xét để xác định các điều chỉnh cho các tham số tinh chỉnh của $T$. Phần dư này có thể biểu diễn là,

$$\epsilon=Y-(P_i(X)+T_i(X))$$

Một thuật toán gradient boosting phổ biến là Xgboost. Các thuật toán này có xu hướng dễ quá khớp và nhạy với nhiễu trong dữ liệu. Kết quả cũng rất nhạy với việc tinh chỉnh tham số.

Đến giờ, ta mới dùng độ chính xác để đánh giá hiệu suất mô hình. Ở phần tiếp theo, ta sẽ xem một chỉ số và biểu đồ thường dùng để đánh giá hiệu suất mô hình.

## **6. Đường cong ROC**

Ta sẽ bàn về một thước đo dùng cho các mô hình phân loại, cụ thể là đường cong ROC (receiver operating curve) và diện tích dưới đường cong (AUC). Các thước đo này thực ra bắt nguồn từ hệ số Gini và đường cong Lorenz, vốn ban đầu dùng để thể hiện bất bình đẳng thu nhập ở các quốc gia. Ta sẽ không tìm hiểu hệ số Gini mà chỉ giữ ứng dụng học máy của nó. Tuy nhiên, nếu bạn đọc quan tâm đến hệ số Gini, xem Gini (1). Trước hết ta trình bày ma trận nhầm lẫn của bộ phân loại nhị phân ở bảng 3, về cơ bản là bảng chéo giữa nhóm dự báo và nhóm thực tế:

**Bảng 3**

| | Thực tế (0) | Thực tế (1) |
|:---|:---:|:---:|
| Dự báo (0) | TN | FN |
| Dự báo (1) | FP | TP |

Các thành phần gồm $TN$ là âm tính thật, tức là các giá trị âm được dự báo đúng hay nhóm 0, còn $FN$ là âm tính giả, tức là các giá trị thực tế nhóm 1 bị dự báo là 0. Tương tự, $TP$ và $FP$ lần lượt là dương tính thật và dương tính giả. Dùng các phần tử trong ma trận nhầm lẫn, ta có tỷ lệ dương tính thật ($TPR$) và tỷ lệ dương tính giả ($FPR$),

$$TPR=\frac{TP}{TP+FN}$$
$$FPR=\frac{FP}{FP+TN}.$$

Cần nhớ rằng có một ngưỡng xác suất gán dự báo vào một nhóm. Ví dụ, nếu ngưỡng là 50% thì mọi dự báo trên 50% sẽ được gán nhóm 1. Do đó $FPR$ và $TPR$ nhạy với ngưỡng.

**Hình 5:** Đường cong ROC của các mô hình khác nhau

![](https://raw.githubusercontent.com/thanhmk/hocmaytrongtaichinh/main/Module_3/M3L3_h5.png)

Nguồn: CMG Lee, MartinThoma, https://commons.wikimedia.org/w/index.php?curid=109730045

ROC là biểu đồ của $TPR$ theo $FPR$ ở các ngưỡng khác nhau và được thể hiện ở hình 5. Mỗi đường hay chuỗi trong biểu đồ đại diện cho một mô hình. Đường đỏ nét đứt là mô hình đoán ngẫu nhiên, còn đường xanh dương là mô hình lý tưởng, dự báo gần như hoàn hảo từng nhóm, tức là các phần tử ngoài đường chéo của ma trận nhầm lẫn gần bằng không. Do đó ta muốn một mô hình càng gần đường xanh dương càng tốt. Đây là lý do đường xanh lá biểu diễn mô hình tốt hơn mô hình được biểu diễn bởi đường cam. ROC cho phép ta nhìn trực quan mô hình nào tốt hơn. Tuy nhiên, đại lượng có thể dùng để so sánh các mô hình hoặc xác định hiệu suất là AUC.

## **7. Diện tích dưới đường cong (AUC)**

AUC đơn giản là diện tích giữa đường cong và trục hoành. AUC càng cao thì mô hình càng phân biệt tốt các nhóm. Bạn vẫn có thể gặp hệ số Gini được dùng để đo hiệu suất của mô hình. Tuy nhiên hãy nhớ rằng AUC và Gini có quan hệ tuyến tính và đồng nghĩa nhau, vì quan hệ do Schechtman và Schechtman (2) thiết lập là,

$$Gini=2*AUC-1.$$

## **8. Kết luận**

Trong bài này, ta đã xem các thuật toán dựa trên cây trong bài toán phân loại và hồi quy. Ta đã xem xét phương pháp luận đằng sau chúng và xem boosting, kết hợp sức mạnh của nhiều bộ học yếu. Một thuật toán dựa trên cây gradient boosting cải tiến cũng được bàn đến như một thước đo để đánh giá hiệu suất mô hình. Ở bài tiếp theo, ta áp dụng các thuật toán dựa trên cây vào một bài toán tài chính.

**Tài liệu tham khảo**

- Gini, Corrado. "On the Measurement of Concentration and Variability of Characters." *Metron - International Journal of Statistics*, vol. 63, no. 1, 2005, pp. 1–38, https://econpapers.repec.org/article/mtnancoec/0501.htm. Originally published in Italian in *Atti del Reale Istituto Veneto di Scienze, Lettere ed Arti*, 1914.

- Hull, Isaiah. *Machine Learning for Economics and Finance in Tensorflow 2.* Apress, 2021. https://link.springer.com/chapter/10.1007/978-1-4842-6373-0_4.

- Schechtman, Edna, and Gideon Schechtman. (2016). "The Relationship between Gini Methodology and the ROC Curve." METRON, vol. 77, 2019, pp. 171–178, https://www.weizmann.ac.il/math/gideon/sites/math.gideon/files/uploads/AUC_Metron%20May%202019%20v4.pdf.